## Importacion de librerias

In [ ]:
import pandas as pd
import numpy as np
import io
import warnings
import re
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import classification_report

print("Listo, librerias cargadas")

## Carga del DATASET

In [ ]:
import zipfile, io

with zipfile.ZipFile('/content/sample_data/steam_games.csv.zip', 'r') as z:
    with z.open('steam_games.csv') as f:
        df_raw = pd.read_csv(f)

print(f"Dataset cargado: {df_raw.shape[0]} filas y {df_raw.shape[1]} columnas")
print(f"Columnas disponibles: {df_raw.columns.tolist()}")
print(f"\nValores faltantes por columna:")
print(df_raw.isnull().sum())

csv_buffer = io.StringIO()
df_raw.to_csv(csv_buffer, index=False)
csv_buffer.seek(0)

df_raw.head(3)

## Primer Agente

In [ ]:
class AgenteNormalizador:

    def __init__(self):
        self.encoders = {}
        self.scaler   = MinMaxScaler()
        self.acciones = []

    def limpiar_texto(self, df):
        for col in df.select_dtypes(include='object').columns:
            df[col] = df[col].str.strip().str.lower()
        self.acciones.append("Texto puesto en minusculas y sin espacios extra")
        return df

    def sacar_outliers(self, df):
        raros = df['temperatura_c'].apply(lambda x: x < 0 or x > 60).sum()
        df['temperatura_c'] = df['temperatura_c'].apply(
            lambda x: np.nan if (x < 0 or x > 60) else x
        )
        self.acciones.append(f"Temperaturas imposibles eliminadas: {raros}")
        return df

    def rellenar_vacios(self, df):
        for col in df.select_dtypes(include=[np.number]).columns:
            vacios = df[col].isnull().sum()
            if vacios > 0:
                mediana = df[col].median()
                df[col] = df[col].fillna(mediana)
                self.acciones.append(f"Columna '{col}': {vacios} vacios rellenados con {mediana:.1f}")
        return df

    def convertir_texto_a_numero(self, df, columnas):
        for col in columnas:
            le = LabelEncoder()
            df[col] = le.fit_transform(df[col].astype(str))
            self.encoders[col] = le
        self.acciones.append(f"Columnas convertidas a numero: {columnas}")
        return df

    def escalar(self, df, columnas):
        df[columnas] = self.scaler.fit_transform(df[columnas])
        self.acciones.append(f"Numeros escalados entre 0 y 1: {columnas}")
        return df

    def ejecutar(self, csv_buffer):
        print("=" * 50)
        print("  AGENTE 1 - NORMALIZADOR")
        print("=" * 50)

        df = pd.read_csv(csv_buffer)
        print(f"Dataset recibido: {df.shape}")

        cols_texto  = ['genre', 'developer', 'types']
        cols_num    = ['original_price', 'discount_price', 'achievements']
        col_destino = 'tiene_descuento'


        def limpiar_precio(val):
            if pd.isna(val): return np.nan
            s_val = str(val).lower()
            if s_val == 'free': return 0.0
            numeros = re.sub(r'[^0-9.]', '', s_val)
            try:
                return float(numeros)
            except ValueError:
                return np.nan

        df['original_price']  = df['original_price'].apply(limpiar_precio)
        df['discount_price']  = df['discount_price'].apply(limpiar_precio)

        df['achievements'] = df['achievements'].fillna(0)

        df['tiene_descuento'] = df['discount_price'].notna().astype(int)

        cols_utiles = ['genre', 'developer', 'types',
                       'original_price', 'discount_price', 'achievements',
                       'tiene_descuento']
        df = df[cols_utiles].copy()

        print(f"Columnas seleccionadas para el modelo: {df.columns.tolist()}")

        df = self.limpiar_texto(df)
        df = self.rellenar_vacios(df)
        df = self.convertir_texto_a_numero(df, cols_texto)
        df = self.escalar(df, cols_num)

        print("\nCosas que hizo el agente:")
        for a in self.acciones:
            print(f"  - {a}")

        print(f"\nDataset limpio listo: {df.shape}")
        return df, col_destino

agente1 = AgenteNormalizador()
csv_buffer.seek(0)
df_limpio, target = agente1.ejecutar(csv_buffer)

print("\nPrimeras filas del dataset limpio:")
df_limpio.head()